In [1]:
import sys
!{sys.executable} -m pip install statsmodels


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import os, sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.seasonal import seasonal_decompose

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 50)
pio.renderers.default = "notebook"

DARK_BG   = "#0d1117"
GOLD      = "#e2b96f"
TEAL      = "#a8dadc"
PURPLE    = "#533483"
BLUE      = "#0f3460"
GREEN     = "#2ecc71"
RED       = "#e74c3c"

print("✅  All libraries loaded successfully.")
print(f"    NumPy  {np.__version__}  |  Pandas  {pd.__version__}")

✅  All libraries loaded successfully.
    NumPy  1.26.4  |  Pandas  3.0.1


In [3]:
df = pd.read_csv("ethereum_usd_historical_dataset.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print("=" * 60)
print("  ETHEREUM USD — KAGGLE DATASET OVERVIEW")
print("=" * 60)
print(f"  Shape        : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date Range   : {df['Date'].iloc[0].date()}  →  {df['Date'].iloc[-1].date()}")
print(f"  All-Time High: ${df['High'].max():,.2f}")
print(f"  All-Time Low : ${df['Low'].min():,.2f}")
print(f"  Latest Close : ${df['Close'].iloc[-1]:,.2f}")
print("=" * 60)
df.head(5)

  ETHEREUM USD — KAGGLE DATASET OVERVIEW
  Shape        : 3,006 rows × 39 columns
  Date Range   : 2017-11-09  →  2026-01-31
  All-Time High: $4,953.73
  All-Time Low : $82.83
  Latest Close : $2,445.09


,Date,Close,High,Low,Open,Volume,Daily_Return_Pct,Log_Return,Price_Range,Body_Size,Upper_Shadow,Lower_Shadow,MA_7,MA_14,MA_21,MA_50,MA_100,MA_200,EMA_12,EMA_26,MACD,Signal_Line,MACD_Histogram,RSI_14,BB_Mid,BB_Std,BB_Upper,BB_Lower,BB_Width,Volume_MA_20,Volume_Change_Pct,Volatility_30d,Year,Month,Quarter,Day_of_Week,Week_Number,Cumulative_Max,Drawdown_Pct
0,2017-11-09,320.88,329.45,307.06,308.64,893249984,NaN,NaN,22.39,12.24,8.57,1.58,NaN,NaN,NaN,NaN,NaN,NaN,320.88,320.88,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017,11,4,Thursday,45,320.88,0.00
1,2017-11-10,299.25,324.72,294.54,320.67,885985984,-6.74,-0.07,30.18,21.42,4.05,4.71,NaN,NaN,NaN,NaN,NaN,NaN,317.55,319.28,-1.73,-0.35,-1.38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.81,NaN,2017,11,4,Friday,45,320.88,-6.74
2,2017-11-11,314.68,319.45,298.19,298.59,842300992,5.16,0.05,21.26,16.09,4.77,0.40,NaN,NaN,NaN,NaN,NaN,NaN,317.11,318.94,-1.83,-0.64,-1.19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.93,NaN,2017,11,4,Saturday,45,320.88,-1.93
3,2017-11-12,307.91,319.15,298.51,314.69,1613479936,-2.15,-0.02,20.64,6.78,4.46,9.40,NaN,NaN,NaN,NaN,NaN,NaN,315.69,318.12,-2.43,-1.00,-1.43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.56,NaN,2017,11,4,Sunday,45,320.88,-4.04
4,2017-11-13,316.72,328.42,307.02,307.02,1041889984,2.86,0.03,21.40,9.70,11.70,0.00,NaN,NaN,NaN,NaN,NaN,NaN,315.85,318.02,-2.17,-1.23,-0.94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-35.43,NaN,2017,11,4,Monday,46,320.88,-1.30


In [4]:
df.tail(5)

,Date,Close,High,Low,Open,Volume,Daily_Return_Pct,Log_Return,Price_Range,Body_Size,Upper_Shadow,Lower_Shadow,MA_7,MA_14,MA_21,MA_50,MA_100,MA_200,EMA_12,EMA_26,MACD,Signal_Line,MACD_Histogram,RSI_14,BB_Mid,BB_Std,BB_Upper,BB_Lower,BB_Width,Volume_MA_20,Volume_Change_Pct,Volatility_30d,Year,Month,Quarter,Day_of_Week,Week_Number,Cumulative_Max,Drawdown_Pct
3001,2026-01-27,3022.21,3031.04,2898.31,2926.20,27976005223,3.27,0.03,132.73,96.01,8.83,27.89,2942.16,3091.05,3106.89,3067.22,3210.96,3671.55,3016.68,3062.27,-45.59,-15.84,-29.75,47.58,3103.89,163.81,3431.51,2776.27,655.24,22295898354.00,-5.71,2.83,2026,1,1,Tuesday,5,4831.35,-37.45
3002,2026-01-28,3006.61,3040.72,2981.59,3022.24,21751566455,-0.52,-0.01,59.13,15.63,18.48,25.02,2946.12,3066.18,3099.26,3060.93,3201.22,3671.87,3015.13,3058.15,-43.02,-21.27,-21.75,46.73,3099.00,165.24,3429.48,2768.52,660.96,22245464782.00,-22.25,2.83,2026,1,1,Wednesday,5,4831.35,-37.77
3003,2026-01-29,2818.23,3008.04,2751.65,3006.26,37487198041,-6.27,-0.06,256.39,188.03,1.78,66.58,2927.34,3030.55,3085.63,3050.79,3190.64,3671.09,2984.84,3040.38,-55.54,-28.13,-27.41,37.91,3085.76,176.80,3439.36,2732.16,707.20,23175848814.00,72.34,3.05,2026,1,1,Thursday,5,4831.35,-41.67
3004,2026-01-30,2702.38,2823.91,2633.84,2818.14,41877689970,-4.11,-0.04,190.07,115.76,5.77,68.54,2891.50,2988.18,3067.50,3040.10,3179.58,3669.54,2941.38,3015.34,-73.96,-37.29,-36.67,33.70,3066.76,196.50,3459.76,2673.76,786.00,24921700164.00,11.71,3.14,2026,1,1,Friday,5,4831.35,-44.07
3005,2026-01-31,2445.09,2709.53,2248.70,2702.30,47569532203,-9.52,-0.10,460.83,257.21,7.23,196.39,2819.55,2926.49,3037.16,3027.31,3165.47,3666.06,2865.03,2973.10,-108.07,-51.45,-56.62,26.63,3033.07,240.03,3513.13,2553.01,960.12,26775315789.00,13.59,3.55,2026,1,1,Saturday,5,4831.35,-49.39


In [5]:
price_cols = ["Open","High","Low","Close","Volume",
              "Daily_Return_Pct","RSI_14","MACD","Volatility_30d"]
df[price_cols].describe().T.style \
    .background_gradient(cmap="YlOrRd", axis=1) \
    .format("{:.2f}") \
    .set_caption("Descriptive Statistics of Key Features")

,count,mean,std,min,25%,50%,75%,max
Open,3006.00,1698.45,1297.30,84.28,354.16,1652.97,2721.93,4831.09
High,3006.00,1744.37,1330.81,85.34,367.37,1687.24,2797.74,4953.73
Low,3006.00,1648.30,1260.40,82.83,348.13,1622.91,2622.19,4718.04
Close,3006.00,1699.02,1296.93,84.31,355.24,1653.84,2722.07,4831.35
Volume,3006.00,15005101859.69,12292076322.32,621732992.00,6326656530.25,12166684523.00,20043340382.00,97736621123.00
Daily_Return_Pct,3005.00,0.17,4.49,-42.35,-1.88,0.06,2.16,26.46
RSI_14,2992.00,51.39,13.63,15.69,41.71,50.09,60.56,90.33
MACD,3006.00,6.37,95.16,-323.65,-31.52,0.51,37.28,454.41
Volatility_30d,2976.00,4.17,1.61,0.81,3.08,3.97,4.96,11.02


In [6]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("No missing values in OHLCV columns.")
    print(f"   Note: {df.isnull().sum().sum()} NaNs total "
          f"(all from rolling-window warm-up rows, by design).")
else:
    print(missing)

MA_200               199
MA_100                99
MA_50                 49
Volatility_30d        30
MA_21                 20
BB_Upper              19
Volume_MA_20          19
BB_Width              19
BB_Lower              19
BB_Mid                19
BB_Std                19
RSI_14                14
MA_14                 13
MA_7                   6
Log_Return             1
Volume_Change_Pct      1
Daily_Return_Pct       1
dtype: int64


In [20]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.75, 0.25],
                    subplot_titles=("ETH/USD Close Price + Moving Averages", "Daily Volume"))

fig.add_trace(go.Scatter(x=df["Date"], y=df["Close"],
                         name="Close", line=dict(color="#e2b96f", width=1.2),
                         opacity=0.9), row=1, col=1)
for ma, col in [("MA_50","#a8dadc"),("MA_200","#ff6b6b"),("MA_21","#82eefd")]:
    fig.add_trace(go.Scatter(x=df["Date"], y=df[ma],
                             name=ma, line=dict(color=col, width=1.5, dash="dot"),
                             opacity=0.8), row=1, col=1)

colors_vol = ["#2ecc71" if c >= o else "#e74c3c"
              for c, o in zip(df["Close"], df["Open"])]
fig.add_trace(go.Bar(x=df["Date"], y=df["Volume"].astype(float),
                     name="Volume", marker_color=colors_vol, opacity=0.6),
              row=2, col=1)

fig.update_layout(
    template="plotly_dark",
    height=600, title_text="Ethereum ETH/USD Full History (2017–2026)",
    title_font=dict(size=18, color="#e2b96f"),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    legend=dict(bgcolor="#0d1117", bordercolor="#333"),
    hovermode="x unified"
)
fig.update_yaxes(title_text="Price (USD)", row=1, col=1, gridcolor="#1e2130")
fig.update_yaxes(title_text="Volume",      row=2, col=1, gridcolor="#1e2130")
fig.update_xaxes(gridcolor="#1e2130", rangeslider_visible=False)
fig.show()


In [19]:
cutoff = df["Date"].max() - pd.DateOffset(months=12)
dfc = df[df["Date"] >= cutoff].copy()

fig = go.Figure(go.Candlestick(
    x=dfc["Date"], open=dfc["Open"], high=dfc["High"],
    low=dfc["Low"],  close=dfc["Close"],
    increasing_line_color=GREEN, decreasing_line_color=RED,
    name="OHLC"
))

fig.add_trace(go.Scatter(x=dfc["Date"], y=dfc["BB_Upper"],
    name="BB Upper", line=dict(color="#a8dadc", dash="dash", width=1), opacity=0.6))
fig.add_trace(go.Scatter(x=dfc["Date"], y=dfc["BB_Lower"],
    name="BB Lower", line=dict(color="#a8dadc", dash="dash", width=1), opacity=0.6,
    fill="tonexty", fillcolor="rgba(168,218,220,0.05)"))

fig.update_layout(
    template="plotly_dark", height=550,
    title="ETH/USD — Candlestick + Bollinger Bands (Last 12 Months)",
    title_font=dict(size=16, color="#e2b96f"),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    xaxis_rangeslider_visible=False
)
fig.show()

In [18]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Daily Return Distribution (KDE)",
                    "30-Day Rolling Volatility"))

ret = df["Daily_Return_Pct"].dropna()
fig.add_trace(go.Histogram(x=ret, nbinsx=120, name="Daily Return %",
    marker_color=GOLD, opacity=0.7,
    histnorm="probability density"), row=1, col=1)

fig.add_trace(go.Scatter(x=df["Date"], y=df["Volatility_30d"],
    name="30d Volatility", line=dict(color=TEAL, width=1.5),
    fill="tozeroy", fillcolor=f"rgba(168,218,220,0.15)"), row=1, col=2)

fig.update_layout(
    template="plotly_dark", height=400,
    title="Daily Return Statistics", title_font=dict(color=GOLD),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    showlegend=False
)
fig.update_xaxes(gridcolor="#1e2130")
fig.update_yaxes(gridcolor="#1e2130")
fig.show()

print(f"   Mean daily return  : {ret.mean():.3f}%")
print(f"   Std  daily return  : {ret.std():.3f}%")
print(f"   Max  daily return  : {ret.max():.2f}%  on {df.loc[df['Daily_Return_Pct']==ret.max(),'Date'].values[0]}")
print(f"   Min  daily return  : {ret.min():.2f}%  on {df.loc[df['Daily_Return_Pct']==ret.min(),'Date'].values[0]}")
print(f"   Positive days      : {(ret>0).sum()} ({100*(ret>0).mean():.1f}%)")
print(f"   Negative days      : {(ret<0).sum()} ({100*(ret<0).mean():.1f}%)")

   Mean daily return  : 0.170%
   Std  daily return  : 4.490%
   Max  daily return  : 26.46%  on 2017-12-12T00:00:00.000000
   Min  daily return  : -42.35%  on 2020-03-12T00:00:00.000000
   Positive days      : 1534 (51.0%)
   Negative days      : 1469 (48.9%)


In [17]:
corr_cols = ["Close","Volume","Daily_Return_Pct","RSI_14",
             "MACD","Volatility_30d","BB_Width","Drawdown_Pct",
             "Price_Range","Body_Size"]
corr = df[corr_cols].corr()

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns.tolist(), y=corr.columns.tolist(),
    colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
    text=np.round(corr.values, 2),
    texttemplate="%{text}",
    colorbar=dict(title="r", tickfont=dict(color="#ccc"))
))
fig.update_layout(
    template="plotly_dark", height=520,
    title="Pearson Correlation Matrix — ETH/USD Features",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    xaxis=dict(tickfont=dict(color="#ccc")),
    yaxis=dict(tickfont=dict(color="#ccc"))
)
fig.show()

In [23]:
annual = df.groupby("Year").agg(
    Open_Price=("Open",  "first"),
    Close_Price=("Close", "last"),
    High=("High","max"),
    Low=("Low","min"),
    Avg_Volume=("Volume","mean")
).reset_index()
annual["Annual_Return_Pct"] = ((annual["Close_Price"] - annual["Open_Price"]) /
                                annual["Open_Price"] * 100).round(2)

colors_yr = [GREEN if v > 0 else RED for v in annual["Annual_Return_Pct"]]
fig = go.Figure(go.Bar(
    x=annual["Year"].astype(str), y=annual["Annual_Return_Pct"],
    marker_color=colors_yr,
    text=[f"{v:.1f}%" for v in annual["Annual_Return_Pct"]],
    textposition="outside", name="Annual Return"
))
fig.update_layout(
    template="plotly_dark", height=400,
    title="Ethereum Annual Return by Year",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    yaxis_title="Return (%)", xaxis_title="Year",
    yaxis=dict(gridcolor="#1e2130", zeroline=True, zerolinecolor="#555")
)
fig.show()
print(annual[["Year","Open_Price","Close_Price","High","Low","Annual_Return_Pct"]].to_string(index=False))

 Year  Open_Price  Close_Price    High     Low  Annual_Return_Pct
 2017      308.64       756.73  881.94  294.54             145.18
 2018      755.76       133.37 1432.88   82.83             -82.35
 2019      133.42       129.61  361.40  102.93              -2.86
 2020      129.63       737.80  754.30   95.18             469.16
 2021      737.71      3682.63 4891.70  718.11             399.20
 2022     3683.05      1196.77 3876.79  896.11             -67.51
 2023     1196.71      2281.47 2445.02 1192.89              90.65
 2024     2282.87      3332.53 4106.96 2113.93              45.98
 2025     3332.41      2967.04 4953.73 1386.80             -10.96
 2026     2967.00      2445.09 3397.90 2248.70             -17.59


In [25]:
monthly = df.pivot_table(index="Year", columns="Month", values="Close", aggfunc="mean")
monthly.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                   "Jul","Aug","Sep","Oct","Nov","Dec"][:len(monthly.columns)]

fig = go.Figure(go.Heatmap(
    z=np.log1p(monthly.values),
    x=monthly.columns.tolist(),
    y=monthly.index.tolist(),
    colorscale="Plasma",
    text=[[f"${v:,.0f}" if not np.isnan(v) else "" for v in row]
          for row in monthly.values],
    texttemplate="%{text}",
    colorbar=dict(title="log(Price)", tickfont=dict(color="#ccc"))
))
fig.update_layout(
    template="plotly_dark", height=420,
    title="Monthly Average Close Price Heatmap (log scale)",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
)
fig.show()

In [27]:
cutoff_ta = df["Date"].max() - pd.DateOffset(months=18)
dft = df[df["Date"] >= cutoff_ta].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.65, 0.35],
                    subplot_titles=("ETH/USD Close Price", "RSI-14"))

fig.add_trace(go.Scatter(x=dft["Date"], y=dft["Close"],
    name="Close", line=dict(color=GOLD, width=1.5)), row=1, col=1)

fig.add_trace(go.Scatter(x=dft["Date"], y=dft["RSI_14"],
    name="RSI-14", line=dict(color=TEAL, width=1.5),
    fill="tozeroy", fillcolor="rgba(168,218,220,0.08)"), row=2, col=1)

for level, col, label in [(70, RED, "Overbought 70"), (30, GREEN, "Oversold 30"), (50, "#888", "Midline 50")]:
    fig.add_hline(y=level, row=2, col=1,
                  line=dict(color=col, dash="dash", width=1),
                  annotation_text=label, annotation_font_color=col)

fig.update_layout(
    template="plotly_dark", height=550,
    title="ETH/USD — RSI-14 (Last 18 Months)",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    hovermode="x unified"
)
fig.update_yaxes(gridcolor="#1e2130")
fig.update_xaxes(gridcolor="#1e2130", rangeslider_visible=False)
fig.show()

In [28]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.6, 0.4],
                    subplot_titles=("Close Price", "MACD (12,26,9)"))

fig.add_trace(go.Scatter(x=dft["Date"], y=dft["Close"],
    name="Close", line=dict(color=GOLD, width=1.5)), row=1, col=1)

fig.add_trace(go.Scatter(x=dft["Date"], y=dft["MACD"],
    name="MACD", line=dict(color=TEAL, width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=dft["Date"], y=dft["Signal_Line"],
    name="Signal", line=dict(color="#ff6b6b", width=1.5, dash="dash")), row=2, col=1)

hist_colors = [GREEN if v >= 0 else RED for v in dft["MACD_Histogram"]]
fig.add_trace(go.Bar(x=dft["Date"], y=dft["MACD_Histogram"],
    name="Histogram", marker_color=hist_colors, opacity=0.6), row=2, col=1)

fig.add_hline(y=0, row=2, col=1, line=dict(color="#555", width=1))

fig.update_layout(
    template="plotly_dark", height=550,
    title="ETH/USD — MACD (12, 26, 9)",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    hovermode="x unified"
)
fig.update_yaxes(gridcolor="#1e2130")
fig.update_xaxes(gridcolor="#1e2130", rangeslider_visible=False)
fig.show()

In [29]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.55, 0.45],
                    subplot_titles=("ETH/USD: Close vs. All-Time High",
                                    "Drawdown from ATH (%)"))

fig.add_trace(go.Scatter(x=df["Date"], y=df["Close"],
    name="Close", line=dict(color=GOLD, width=1), opacity=0.9), row=1, col=1)
fig.add_trace(go.Scatter(x=df["Date"], y=df["Cumulative_Max"],
    name="ATH", line=dict(color="#aaa", width=1, dash="dot"), opacity=0.7), row=1, col=1)

fig.add_trace(go.Scatter(x=df["Date"], y=df["Drawdown_Pct"],
    name="Drawdown %", line=dict(color=RED, width=1.2),
    fill="tozeroy", fillcolor=f"rgba(231,76,60,0.15)"), row=2, col=1)

# Label worst drawdown
worst_idx = df["Drawdown_Pct"].idxmin()
fig.add_annotation(
    x=df.loc[worst_idx, "Date"], y=df.loc[worst_idx, "Drawdown_Pct"],
    text=f"Worst: {df.loc[worst_idx,'Drawdown_Pct']:.1f}%",
    showarrow=True, arrowhead=2, row=2, col=1,
    font=dict(color="white"), bgcolor=RED, bordercolor=RED
)

fig.update_layout(
    template="plotly_dark", height=560,
    title="ETH/USD — Drawdown from All-Time High",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    hovermode="x unified"
)
fig.update_yaxes(gridcolor="#1e2130")
fig.update_xaxes(gridcolor="#1e2130", rangeslider_visible=False)
fig.show()

In [30]:
df_dec = df[df["Date"] >= "2021-01-01"].copy().set_index("Date")["Close"].ffill()
result = seasonal_decompose(df_dec, model="multiplicative", period=365)

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
    subplot_titles=["Observed", "Trend", "Seasonal", "Residual"])

for i, (name, series) in enumerate([("Observed",   result.observed),
                                      ("Trend",      result.trend),
                                      ("Seasonal",   result.seasonal),
                                      ("Residual",   result.resid)], start=1):
    fig.add_trace(
        go.Scatter(x=series.index, y=series.values,
                   name=name, mode="lines",
                   line=dict(color=[GOLD, TEAL, "#82eefd", RED][i-1], width=1.2)),
        row=i, col=1
    )

fig.update_layout(
    template="plotly_dark", height=700,
    title="Multiplicative Seasonal Decomposition (2021–2026)",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    showlegend=False
)
fig.update_yaxes(gridcolor="#1e2130")
fig.update_xaxes(gridcolor="#1e2130")
fig.show()

In [31]:
LAGS    = [1, 2, 3, 5, 7, 10, 14, 21, 30]
TARGETS = [1, 7, 14, 30]   

df_ml = df.copy()

for lag in LAGS:
    df_ml[f"lag_{lag}"] = df_ml["Close"].shift(lag)

for lag in [1, 2, 3, 5, 7]:
    df_ml[f"ret_lag_{lag}"] = df_ml["Daily_Return_Pct"].shift(lag)

for w in [7, 14, 30]:
    df_ml[f"roll_mean_{w}"] = df_ml["Close"].rolling(w).mean()
    df_ml[f"roll_std_{w}"]  = df_ml["Close"].rolling(w).std()
    df_ml[f"roll_min_{w}"]  = df_ml["Close"].rolling(w).min()
    df_ml[f"roll_max_{w}"]  = df_ml["Close"].rolling(w).max()

tech_features = ["RSI_14","MACD","Signal_Line","MACD_Histogram",
                 "BB_Width","Volatility_30d","Price_Range",
                 "EMA_12","EMA_26","MA_50","MA_200"]

lag_cols  = [f"lag_{l}"     for l in LAGS]
rlag_cols = [f"ret_lag_{l}" for l in [1,2,3,5,7]]
roll_cols = ([f"roll_mean_{w}" for w in [7,14,30]] +
             [f"roll_std_{w}"  for w in [7,14,30]] +
             [f"roll_min_{w}"  for w in [7,14,30]] +
             [f"roll_max_{w}"  for w in [7,14,30]])

FEATURE_COLS = lag_cols + rlag_cols + roll_cols + tech_features
TARGET_COL   = "Close"

df_ml.dropna(subset=FEATURE_COLS + [TARGET_COL], inplace=True)
df_ml = df_ml.sort_values("Date").reset_index(drop=True)

print(f"ML dataset: {df_ml.shape[0]} rows × {len(FEATURE_COLS)} features")
print(f"Training cutoff  : {df_ml['Date'].iloc[-1].date()}")

ML dataset: 2807 rows × 37 features
Training cutoff  : 2026-01-31


In [32]:
TRAIN_END = "2025-09-30"

train = df_ml[df_ml["Date"] <= TRAIN_END].copy()
test  = df_ml[df_ml["Date"] >  TRAIN_END].copy()

X_train, y_train = train[FEATURE_COLS].values, train[TARGET_COL].values
X_test,  y_test  = test[FEATURE_COLS].values,  test[TARGET_COL].values

print(f"Train : {len(train):,} rows  ({train['Date'].iloc[0].date()} → {train['Date'].iloc[-1].date()})")
print(f"Test  : {len(test):,}  rows  ({test['Date'].iloc[0].date()} → {test['Date'].iloc[-1].date()})")

xgb_model = XGBRegressor(
    n_estimators=800, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

lgb_model = LGBMRegressor(
    n_estimators=800, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1, verbose=-1
)
lgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              callbacks=[])

print("\n Both models trained.")

Train : 2,684 rows  (2018-05-27 → 2025-09-30)
Test  : 123  rows  (2025-10-01 → 2026-01-31)

 Both models trained.


In [33]:
def evaluate(name, y_true, y_pred):
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mape  = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2    = r2_score(y_true, y_pred)
    return {"Model": name, "MAE": mae, "RMSE": rmse, "MAPE (%)": mape, "R²": r2}

xgb_pred = xgb_model.predict(X_test)
lgb_pred = lgb_model.predict(X_test)
ens_pred = (xgb_pred + lgb_pred) / 2   
results = pd.DataFrame([
    evaluate("XGBoost",   y_test, xgb_pred),
    evaluate("LightGBM",  y_test, lgb_pred),
    evaluate("Ensemble",  y_test, ens_pred),
])
results = results.set_index("Model")

styled = results.style \
    .background_gradient(cmap="Greens", subset=["R²"]) \
    .background_gradient(cmap="Reds_r",  subset=["MAE","RMSE","MAPE (%)"]) \
    .format({"MAE": "${:.2f}", "RMSE": "${:.2f}", "MAPE (%)": "{:.2f}%", "R²": "{:.4f}"}) \
    .set_caption(" Model Evaluation on Hold-out Test Set (Oct–Jan 2026)")

print("\n EVALUATION RESULTS")
print(results.round(4).to_string())
styled


 EVALUATION RESULTS
           MAE  RMSE  MAPE (%)   R²
Model                              
XGBoost  45.30 62.73      1.36 0.98
LightGBM 49.25 64.73      1.47 0.98
Ensemble 45.11 60.95      1.35 0.99


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



,MAE,RMSE,MAPE (%),R²
Model,,,,
XGBoost,$45.30,$62.73,1.36%,0.9844
LightGBM,$49.25,$64.73,1.47%,0.9834
Ensemble,$45.11,$60.95,1.35%,0.9853


In [34]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=test["Date"], y=y_test,
    name="Actual", line=dict(color=GOLD, width=2)))
fig.add_trace(go.Scatter(x=test["Date"], y=xgb_pred,
    name="XGBoost", line=dict(color=TEAL, width=1.5, dash="dot")))
fig.add_trace(go.Scatter(x=test["Date"], y=lgb_pred,
    name="LightGBM", line=dict(color="#82eefd", width=1.5, dash="dash")))
fig.add_trace(go.Scatter(x=test["Date"], y=ens_pred,
    name="Ensemble", line=dict(color="#ff6b6b", width=2)))

fig.update_layout(
    template="plotly_dark", height=480,
    title="ETH/USD — Actual vs. Model Predictions (Test Period)",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    hovermode="x unified",
    legend=dict(bgcolor="#0d1117", bordercolor="#333")
)
fig.update_yaxes(title_text="Price (USD)", gridcolor="#1e2130")
fig.update_xaxes(gridcolor="#1e2130")
fig.show()

In [35]:
fi_xgb = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)[:20]
fi_lgb = pd.Series(lgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)[:20]
fig = make_subplots(rows=1, cols=2, subplot_titles=("XGBoost Top-20 Features",
                                                      "LightGBM Top-20 Features"))
fig.add_trace(go.Bar(y=fi_xgb.index, x=fi_xgb.values, orientation="h",
    marker_color=GOLD, name="XGBoost"), row=1, col=1)
fig.add_trace(go.Bar(y=fi_lgb.index, x=fi_lgb.values, orientation="h",
    marker_color=TEAL, name="LightGBM"), row=1, col=2)
fig.update_layout(
    template="plotly_dark", height=550,
    title="Feature Importance — Top 20",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    showlegend=False
)
fig.update_xaxes(gridcolor="#1e2130")
fig.update_yaxes(autorange="reversed", tickfont=dict(size=10))
fig.show()

In [37]:
import datetime
def recursive_forecast(model_xgb, model_lgb, df_full, feature_cols, horizon=183):
    history = df_full[["Date","Close"] + feature_cols].copy()
    history = history.sort_values("Date").reset_index(drop=True)

    last_date  = history["Date"].iloc[-1]
    forecast_rows = []
    close_series = list(history["Close"].values)
    ret_series   = list(df_full["Daily_Return_Pct"].fillna(0).values)

    for step in range(1, horizon + 1):
        next_date = last_date + pd.Timedelta(days=step)
        cs  = close_series
        rs  = ret_series
        n   = len(cs)

        lag_vals  = {f"lag_{l}":    cs[-(l)]     if n >= l    else cs[0] for l in LAGS}
        rlag_vals = {f"ret_lag_{l}": rs[-(l)]    if n >= l    else 0     for l in [1,2,3,5,7]}

        def roll_safe(arr, w, fn):
            window = arr[-w:] if len(arr) >= w else arr
            return fn(window)

        roll_vals = {}
        for w in [7, 14, 30]:
            roll_vals[f"roll_mean_{w}"] = roll_safe(cs, w, np.mean)
            roll_vals[f"roll_std_{w}"]  = roll_safe(cs, w, np.std)
            roll_vals[f"roll_min_{w}"]  = roll_safe(cs, w, np.min)
            roll_vals[f"roll_max_{w}"]  = roll_safe(cs, w, np.max)

        last_row = history.iloc[-1]
        tech_vals = {col: float(last_row[col]) if col in history.columns else 0.0
                     for col in tech_features}

        feat_dict = {**lag_vals, **rlag_vals, **roll_vals, **tech_vals}
        X_fut = np.array([[feat_dict[c] for c in feature_cols]])

        p_xgb = float(model_xgb.predict(X_fut)[0])
        p_lgb = float(model_lgb.predict(X_fut)[0])
        p_ens = (p_xgb + p_lgb) / 2
        daily_ret = (p_ens / close_series[-1] - 1) * 100

        forecast_rows.append({
            "Date":    next_date,
            "XGBoost": round(p_xgb, 2),
            "LightGBM":round(p_lgb, 2),
            "Ensemble":round(p_ens, 2),
        })

        close_series.append(p_ens)
        ret_series.append(daily_ret)

    return pd.DataFrame(forecast_rows)
forecast_horizon = 150   
print("Generating recursive 150-day forecast …")
forecast_df = recursive_forecast(xgb_model, lgb_model, df_ml, FEATURE_COLS, horizon=forecast_horizon)

forecast_df["Quarter"] = forecast_df["Date"].dt.quarter
forecast_df["Month_Name"] = forecast_df["Date"].dt.strftime("%b %Y")

q1_fc = forecast_df[forecast_df["Date"].dt.month.isin([2,3])].copy()   
q2_fc = forecast_df[forecast_df["Date"].dt.month.isin([4,5,6])].copy() 

print(f"Forecast generated: {len(forecast_df)} days")
print(f"Q1 2026 (Feb-Mar): {len(q1_fc)} days")
print(f"Q2 2026 (Apr-Jun): {len(q2_fc)} days")
forecast_df.head()

Generating recursive 150-day forecast …
Forecast generated: 150 days
Q1 2026 (Feb-Mar): 59 days
Q2 2026 (Apr-Jun): 91 days


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning:

X does not have valid fe

,Date,XGBoost,LightGBM,Ensemble,Quarter,Month_Name
0,2026-02-01,2419.93,2371.81,2395.87,1,Feb 2026
1,2026-02-02,2399.39,2333.30,2366.34,1,Feb 2026
2,2026-02-03,2355.63,2295.37,2325.50,1,Feb 2026
3,2026-02-04,2244.18,2238.96,2241.57,1,Feb 2026
4,2026-02-05,2229.48,2199.62,2214.55,1,Feb 2026


In [39]:
look_back = df_ml[df_ml["Date"] >= "2025-11-01"].copy()

ci_upper = (forecast_df["Ensemble"] * 1.15).values
ci_lower = (forecast_df["Ensemble"] * 0.85).values

fig = go.Figure()

dates_fwd = forecast_df["Date"].reset_index(drop=True)
dates_rev = forecast_df["Date"].reset_index(drop=True)[::-1].reset_index(drop=True)
fig.add_trace(go.Scatter(
    x=pd.concat([dates_fwd, dates_rev], ignore_index=True),
    y=np.concatenate([ci_upper, ci_lower[::-1]]),
    fill="toself", fillcolor="rgba(168,218,220,0.10)",
    line=dict(color="rgba(0,0,0,0)"), showlegend=True, name="±15% Confidence Band"
))

fig.add_trace(go.Scatter(x=look_back["Date"], y=look_back["Close"],
    name="Actual (Historic)", line=dict(color=GOLD, width=2)))

fig.add_trace(go.Scatter(x=forecast_df["Date"], y=forecast_df["XGBoost"],
    name="XGBoost", line=dict(color="#82eefd", width=1.5, dash="dot"),
    opacity=0.7))

fig.add_trace(go.Scatter(x=forecast_df["Date"], y=forecast_df["LightGBM"],
    name="LightGBM", line=dict(color="#ff6b6b", width=1.5, dash="dash"),
    opacity=0.7))

fig.add_trace(go.Scatter(x=forecast_df["Date"], y=forecast_df["Ensemble"],
    name="Ensemble Forecast", line=dict(color=TEAL, width=2.5)))

for m, label in [(3, "Q1/Q2 Boundary"), (6, "Q2 End")]:
    boundary = forecast_df[forecast_df["Date"].dt.month == m]["Date"].max()
    if pd.notna(boundary):
        boundary_ms = int(pd.Timestamp(boundary).timestamp() * 1000)
        fig.add_vline(x=boundary_ms,
                      line=dict(color="#555", dash="dash", width=1),
                      annotation_text=label, annotation_font_color="#aaa")
fig.update_layout(
    template="plotly_dark", height=560,
    title="ETH/USD — Q1 & Q2 2026 Price Forecast (Ensemble Model)",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    hovermode="x unified",
    legend=dict(bgcolor="#0d1117", bordercolor="#333")
)
fig.update_yaxes(title_text="Price (USD)", gridcolor="#1e2130")
fig.update_xaxes(gridcolor="#1e2130")
fig.show()

In [41]:
monthly_fc = forecast_df.groupby(forecast_df["Date"].dt.to_period("M")).agg(
    Predicted_Open  = ("Ensemble", "first"),
    Predicted_High  = ("Ensemble", "max"),
    Predicted_Low   = ("Ensemble", "min"),
    Predicted_Close = ("Ensemble", "last"),
    Avg_Price       = ("Ensemble", "mean"),
).reset_index()
monthly_fc["Date"] = monthly_fc["Date"].astype(str)
monthly_fc = monthly_fc.round(2)

print("=" * 70)
print("  MONTHLY FORECAST SUMMARY — Q1 & Q2 2026 (ETH/USD)")
print("=" * 70)
print(monthly_fc.to_string(index=False))
print("=" * 70)

fig = px.bar(monthly_fc, x="Date", y="Avg_Price",
             color="Avg_Price", color_continuous_scale="Plasma",
             text="Avg_Price",
             title="Monthly Average Forecasted Price — ETH/USD (Q1-Q2 2026)")
fig.update_traces(texttemplate="$%{text:,.0f}", textposition="outside")
fig.update_layout(
    template="plotly_dark", height=420,
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    title_font=dict(color=GOLD, size=16),
    yaxis=dict(gridcolor="#1e2130", title="USD"),
    xaxis_title="Month"
)
fig.show()

  MONTHLY FORECAST SUMMARY — Q1 & Q2 2026 (ETH/USD)
   Date  Predicted_Open  Predicted_High  Predicted_Low  Predicted_Close  Avg_Price
2026-02         2395.87         2395.87        1509.67          1509.67    1887.01
2026-03         1497.60         1497.60        1124.26          1124.26    1391.21
2026-04         1114.77         1114.77        1061.90          1064.65    1073.18
2026-05         1064.67         1064.80        1064.22          1064.63    1064.48
2026-06         1064.33         1064.63        1064.33          1064.63    1064.48


In [42]:
print("\n Q1 2026 Summary (Feb - Mar)")
q1_summary = q1_fc["Ensemble"].agg(["min","mean","max","std"])
for k,v in q1_summary.items():
    print(f"   {k.capitalize():10} : ${v:,.2f}")

print("\n Q2 2026 Summary (Apr - Jun)")
q2_summary = q2_fc["Ensemble"].agg(["min","mean","max","std"])
for k,v in q2_summary.items():
    print(f"   {k.capitalize():10} : ${v:,.2f}")

current_price = df["Close"].iloc[-1]
q1_avg = q1_fc["Ensemble"].mean()
q2_avg = q2_fc["Ensemble"].mean()
print(f"\n Jan 31, 2026 Close  : ${current_price:,.2f}")
print(f"   Q1 2026 Avg Forecast: ${q1_avg:,.2f}  |  Change: {(q1_avg/current_price-1)*100:+.1f}%")
print(f"   Q2 2026 Avg Forecast: ${q2_avg:,.2f}  |  Change: {(q2_avg/current_price-1)*100:+.1f}%")


 Q1 2026 Summary (Feb - Mar)
   Min        : $1,124.26
   Mean       : $1,626.50
   Max        : $2,395.87
   Std        : $319.24

 Q2 2026 Summary (Apr - Jun)
   Min        : $1,061.90
   Mean       : $1,067.35
   Max        : $1,114.77
   Std        : $9.73

 Jan 31, 2026 Close  : $2,445.09
   Q1 2026 Avg Forecast: $1,626.50  |  Change: -33.5%
   Q2 2026 Avg Forecast: $1,067.35  |  Change: -56.3%


In [43]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pd.concat([forecast_df["Date"], forecast_df["Date"][::-1]]),
    y=np.concatenate([forecast_df["Ensemble"] * 1.30,
                      (forecast_df["Ensemble"] * 0.70)[::-1]]),
    fill="toself", fillcolor="rgba(83,52,131,0.15)",
    line=dict(color="rgba(0,0,0,0)"),
    name="Bull-Bear Range", showlegend=True
))
fig.add_trace(go.Scatter(x=look_back["Date"], y=look_back["Close"],
    name="Actual Close", line=dict(color=GOLD, width=2.5)))

fig.add_trace(go.Scatter(x=forecast_df["Date"], y=forecast_df["Ensemble"],
    name="Base (Ensemble)", line=dict(color=TEAL, width=2.5)))

fig.add_trace(go.Scatter(x=forecast_df["Date"],
    y=(forecast_df["Ensemble"] * 1.30),
    name="Bull (+30%)", line=dict(color=GREEN, width=1.5, dash="dot")))

fig.add_trace(go.Scatter(x=forecast_df["Date"],
    y=(forecast_df["Ensemble"] * 0.70),
    name="Bear (-30%)", line=dict(color=RED, width=1.5, dash="dot")))

fig.update_layout(
    template="plotly_dark", height=520,
    title="ETH/USD — Q1 & Q2 2026 Scenario Analysis: Bull / Base / Bear",
    title_font=dict(color=GOLD, size=16),
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    hovermode="x unified",
    legend=dict(bgcolor="#0d1117", bordercolor="#333")
)
fig.update_yaxes(title_text="Price (USD)", gridcolor="#1e2130")
fig.update_xaxes(gridcolor="#1e2130")
fig.show()